# Solar Maximum Lab

Audience: learners and systems engineers who want to inspect a deterministic solar-surface model without treating it as an operational forecast.

Prerequisites: basic Python, JSON, and comfort reading small arrays.

Learning goals:
- Load a versioned `solar-state-snapshot.v1` fixture.
- Identify synthetic, inferred, observed, and degraded layers.
- Connect active regions to the solar maximum stage.
- Inspect provenance before trusting an observed layer.


## Outline

1. Load the deterministic snapshot.
2. Check the contract and field dimensions.
3. Summarize magnetic activity and confidence.
4. Inspect observation provenance.
5. Try a small exercise on layer labels.


In [ ]:
from pathlib import Path
import json

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "apps/web/data/latest-state.json").exists():
            return candidate
    raise FileNotFoundError("apps/web/data/latest-state.json")

REPO_ROOT = find_repo_root(Path.cwd())
SNAPSHOT_PATH = REPO_ROOT / "apps/web/data/latest-state.json"
with SNAPSHOT_PATH.open("r", encoding="utf-8") as handle:
    snapshot = json.load(handle)

print(snapshot["schema_version"])
print(snapshot["learning"]["cycle_stage"])
print(snapshot["learning"]["plain_language_insight"])


## Contract Check

The renderer expects immutable snapshots. The grid length must match every scalar field that is drawn on the solar disk.


In [ ]:
grid = snapshot["grid"]
expected_len = grid["lon_count"] * grid["lat_count"]
field_lengths = {
    name: len(payload["values"])
    for name, payload in snapshot["fields"].items()
    if "values" in payload
}

print({"expected_len": expected_len, "field_lengths": field_lengths})
assert all(length == expected_len for length in field_lengths.values())
assert snapshot["operational_use"] is False


## Activity Summary

A solar maximum run should show many active regions and a broad distribution of magnetic-field strength. Values remain normalized until calibrated physical units are implemented.


In [ ]:
br = snapshot["fields"]["br_normalized"]["values"]
confidence = snapshot["fields"]["confidence"]["values"]
regions = snapshot["active_regions"]
summary = {
    "active_regions": len(regions),
    "max_abs_br": round(max(abs(value) for value in br), 4),
    "mean_confidence": round(sum(confidence) / len(confidence), 4),
    "source_mode": snapshot["source_mode"],
}
print(summary)


## Provenance Inspection

Observed context is useful only when the source, timestamp, and quality flags remain visible. Fixture-backed rows should be treated differently from live cached rows.


In [ ]:
for report in snapshot.get("observations", []):
    print("report", report["schema_version"], report["source_mode"])
    for frame in report.get("frames", []):
        provenance = frame.get("provenance", {})
        print(frame["id"], provenance.get("time_tag"), provenance.get("source"), provenance.get("active"), frame.get("quality_flags"))


## Exercise

Build a dictionary that maps each layer id to its disclosure label. Confirm no layer uses a label outside `synthetic`, `observed`, `blended`, `inferred`, or `degraded`.


In [ ]:
allowed = {"synthetic", "observed", "blended", "inferred", "degraded"}
layer_labels = {layer["id"]: layer["kind"] for layer in snapshot["layers"]}
print(layer_labels)
assert set(layer_labels.values()).issubset(allowed)


## Pitfall And Extension

Pitfall: treating fixture or synthetic values as calibrated observations. Fix it by checking `source_mode`, layer `kind`, and provenance before interpreting the result.

Extension: run `tools/fetch_public_data.py --cache .cache/solar-data`, then compare cached SWPC context against the deterministic fixture output.
